# Matemáticas para la Inteligencia Artificial
## Ejercicios Entregables

---

**Asignatura:** 02MIAR_04_A_2026-27 — Matemáticas para la Inteligencia Artificial  
**Grupo:** Nuevo Grupo 8  
**Curso:** 2026-27  

---

### Integrantes del grupo

| # | Nombre |
|:-:|:-------|
| 1 | Adelso Steve Araya Solorzano |
| 2 | Fabio Antonio Rodriguez Quesada |

---

In [15]:
import numpy as np
import sympy as sp
from numpy.polynomial import Polynomial
import math

---

## Ejercicio 1 — Cálculo del Determinante

Tal y como ya hemos visto en clase, la variedad de herramientas proporcionadas por el álgebra lineal son cruciales para desarrollar y fundamentar las bases de una variedad de técnicas relacionadas con el aprendizaje automático. Con ella, podemos describir el proceso de propagación hacia adelante en una red neuronal, identificar mínimos locales en funciones multivariables (crucial para el proceso de retropropagación) o la descripción y empleo de métodos de reducción de la dimensionalidad, como el análisis de componentes principales (PCA), entre muchas otras aplicaciones.

Cuando trabajamos en la práctica dentro de este ámbito, la cantidad de datos que manejamos puede ser muy grande, por lo que es especialmente importante emplear algoritmos eficientes y optimizados para reducir el coste computacional en la medida de lo posible. Por todo ello, el objetivo de este ejercicio es el de ilustrar las diferentes alternativas que pueden existir para realizar un proceso relacionado con el álgebra lineal y el impacto que puede tener cada variante en términos del coste computacional del mismo. En este caso en particular, y a modo de ilustración, nos centraremos en el cálculo del determinante de una matriz.

**a) [1 punto]** Implementa una función, `determinante_recursivo`, que obtenga el determinante de una matriz cuadrada utilizando la definición recursiva de Laplace.

In [2]:
def eliminar_fila_columna(matriz, fila, columna):
    matriz_secundaria = matriz.copy()
    matriz_secundaria = np.delete(matriz_secundaria, fila, axis=0)
    matriz_secundaria = np.delete(matriz_secundaria, columna, axis=1)
    return matriz_secundaria


def determinante_recursivo(matriz):
    if len(matriz) == 1:
        return matriz[0][0]

    if len(matriz) == 2:
        return matriz[0][0] * matriz[1][1] - matriz[0][1] * matriz[1][0]

    determinante = 0
    for i in range(len(matriz)):
        signo = (-1) ** i
        pivote = matriz[0][i]
        submatriz = eliminar_fila_columna(matriz, 0, i)
        determinante += signo * pivote * determinante_recursivo(submatriz)
    return determinante


matriz = np.array([[4, 2, -1], [3, 5, 2], [-1, 0, 7]])
print(f"Determinante de la matriz: {determinante_recursivo(matriz)}")

Determinante de la matriz: 89


**b) [0.5 puntos]** Si $A$ es una matriz cuadrada $n \times n$ y triangular (superior o inferior, es decir, con entradas nulas por debajo o por encima de la diagonal, respectivamente), ¿existe alguna forma de calcular de forma directa y sencilla su determinante? Justifíquese la respuesta.

> **R/**
>
> Sí. En una matriz triangular (superior o inferior), al expandir el determinante por la definición de Laplace, todos los cofactores que involucran elementos fuera de la diagonal principal se anulan, ya que dichos elementos son cero. Por lo tanto, el único término que sobrevive en cada expansión es el correspondiente al elemento de la diagonal, lo que lleva a que el determinante sea simplemente el producto de los elementos de la diagonal principal:
>
> $$\det(A) = \prod_{i=1}^{n} a_{ii} = a_{11} \cdot a_{22} \cdots a_{nn}$$
>
> Esto aplica tanto para matrices triangulares superiores como inferiores, y también para matrices diagonales como caso particular.

**Ejemplo práctico:**

In [3]:
# Matriz triangular superior
A = np.array([[3, 5, 2],
              [0, 4, 1],
              [0, 0, 2]])

det_diagonal = np.prod(np.diag(A))
det_numpy    = np.linalg.det(A)

print(f"Producto de la diagonal: {det_diagonal}")
print(f"Determinante (numpy):    {det_numpy:.1f}")

Producto de la diagonal: 24
Determinante (numpy):    24.0


**c) [0.5 puntos]** Determínese de forma justificada cómo alteran el determinante de una matriz $n \times n$ las dos operaciones elementales siguientes:

- Intercambiar una fila (o columna) por otra fila (o columna).
- Sumar a una fila (o columna) otra fila (o columna) multiplicada por un escalar $\alpha$.

> **R/**
>
> **A)** Intercambiar dos filas (o columnas) de una matriz multiplica su determinante por $-1$. Esto se debe a que el determinante es una función alternada en sus filas y columnas: cualquier intercambio de dos filas o columnas, sean adyacentes o no, invierte el signo del determinante.
>
> $$\det(A') = -\det(A)$$
>
> **B)** Sumar a una fila (o columna) otra fila (o columna) multiplicada por un escalar $\alpha$ **no altera** el determinante. Esto se debe a la linealidad del determinante: el término adicional genera una matriz con dos filas (o columnas) proporcionales, cuyo determinante es cero, por lo que la suma no modifica el valor original.
>
> $$\det(A') = \det(A)$$

**Ejemplo práctico:**

In [4]:
A = np.array([[4, 2, -1], [3, 5, 2], [-1, 0, 7]], dtype=float)

# A) Intercambiar filas 0 y 1
A_intercambiada = A[[1, 0, 2], :]
print(f"det(A)              = {np.linalg.det(A):.2f}")
print(f"det(A intercambiada)= {np.linalg.det(A_intercambiada):.2f}")
print()

# B) Sumar a fila 1 la fila 0 multiplicada por alpha=2
alpha = 2
A_sumada = A.copy()
A_sumada[1] = A_sumada[1] + alpha * A_sumada[0]
print(f"det(A)       = {np.linalg.det(A):.2f}")
print(f"det(A sumada)= {np.linalg.det(A_sumada):.2f}")

det(A)              = 89.00
det(A intercambiada)= -89.00

det(A)       = 89.00
det(A sumada)= 89.00


**d) [1 punto]** Investiga sobre el método de eliminación de Gauss con pivoteo parcial e impleméntalo para escalonar una matriz (es decir, convertirla en una matriz triangular inferior) a partir de las operaciones elementales descritas en el apartado anterior.

In [5]:
def eliminacion_gauss_pivoteo(matriz):
    A = matriz.copy().astype(float)
    n = len(A)

    for col in range(n - 1, -1, -1):
        # Pivoteo parcial: intercambiar con la fila de mayor valor absoluto
        fila_max = np.argmax(np.abs(A[:col + 1, col]))
        if fila_max != col:
            A[[col, fila_max]] = A[[fila_max, col]]

        # Eliminación: hacer ceros por encima del pivote
        for fila in range(col - 1, -1, -1):
            if A[col][col] == 0:
                continue
            multiplicador = A[fila][col] / A[col][col]
            A[fila] = A[fila] - multiplicador * A[col]

    return A


# Ejemplo de uso
matriz = np.array([[4, 2, -1],
                   [3, 5,  2],
                   [-1, 0,  7]], dtype=float)

resultado = eliminacion_gauss_pivoteo(matriz)
print("Matriz triangular inferior:")
print(np.round(resultado, 5))

Matriz triangular inferior:
[[ 2.54286  0.       0.     ]
 [ 3.28571  5.       0.     ]
 [-1.       0.       7.     ]]


**e) [0.5 puntos]** ¿Cómo se podría calcular el determinante de una matriz haciendo beneficio de la estrategia anterior y del efecto de aplicar las operaciones elementales pertinentes? Implementa una nueva función, `determinante_gauss`, que calcule el determinante de una matriz utilizando eliminación gaussiana.

In [6]:
def determinante_gauss(matriz):
    A = matriz.copy().astype(float)
    n = len(A)
    signo = 1

    # Rastrear intercambios de filas para ajustar el signo del determinante
    for col in range(n - 1, -1, -1):
        fila_max = np.argmax(np.abs(A[:col + 1, col]))
        if fila_max != col:
            signo *= -1

    # Usar la función del apartado d) para obtener la matriz triangular inferior
    A_triangular = eliminacion_gauss_pivoteo(matriz)

    # El determinante es el producto de la diagonal ajustado por el signo
    return signo * np.prod(np.diag(A_triangular))


# Verificación con la misma matriz de 1a)
matriz = np.array([[4, 2, -1,],
                   [3, 5,  2],
                   [-1, 0,  7]], dtype=float)

print(f"Determinante (Gauss):     {determinante_gauss(matriz):.2f}")
print(f"Determinante (numpy):     {np.linalg.det(matriz):.2f}")
print(f"Determinante (recursivo): {determinante_recursivo(matriz)}")

Determinante (Gauss):     89.00
Determinante (numpy):     89.00
Determinante (recursivo): 89.0


**f) [0.5 puntos]** Obtén la complejidad computacional asociada al cálculo del determinante con la definición recursiva y con el método de eliminación de Gauss con pivoteo parcial.

> **R/**
>
> **Definición recursiva de Laplace — $O(n!)$**
>
> En cada llamada recursiva, se expande por $n$ cofactores, y cada uno genera una submatriz de tamaño $(n-1) \times (n-1)$. El número total de operaciones sigue la recurrencia:
>
> $$T(n) = n \cdot T(n-1) \implies T(n) = O(n!)$$
>
> Esto lo hace inviable para matrices grandes, ya que el coste crece factorialmente.
>
> **Eliminación de Gauss con pivoteo parcial — $O(n^3)$**
>
> El algoritmo realiza $n$ iteraciones externas (una por columna). En cada iteración se busca el pivote en $O(n)$ y se eliminan los elementos de $n$ filas, cada una con $n$ operaciones. Por tanto:
>
> $$T(n) = n \cdot n \cdot n = O(n^3)$$
>
> Esto representa una mejora drástica frente al método recursivo, haciendo viable el cálculo para matrices de gran tamaño.

**g) [1 punto]** Utilizando `numpy.random.rand`, genera matrices cuadradas aleatorias de la forma $A_n \in \mathbb{R}^{n \times n}$, para $2 \leq n \leq 10$, y confecciona una tabla comparativa del tiempo de ejecución asociado a cada una de las variantes siguientes, interpretando los resultados:

- Utilizando `determinante_recursivo`.
- Empleando `determinante_gauss`.
- Haciendo uso de la función preprogramada `numpy.linalg.det`.

In [7]:
import time

print(f"{'n':>4} | {'Recursivo (s)':>15} | {'Gauss (s)':>12} | {'NumPy (s)':>12}")
print("-" * 52)

for n in range(2, 11):
    A = np.random.rand(n, n)

    t0 = time.perf_counter()
    for _ in range(10): determinante_recursivo(A)
    t_recursivo = (time.perf_counter() - t0) / 10

    t0 = time.perf_counter()
    for _ in range(10): determinante_gauss(A)
    t_gauss = (time.perf_counter() - t0) / 10

    t0 = time.perf_counter()
    for _ in range(10): np.linalg.det(A)
    t_numpy = (time.perf_counter() - t0) / 10

    print(f"{n:>4} | {t_recursivo:>15.6f} | {t_gauss:>12.6f} | {t_numpy:>12.6f}")

   n |   Recursivo (s) |    Gauss (s) |    NumPy (s)
----------------------------------------------------
   2 |        0.000002 |     0.000078 |     0.000010
   3 |        0.000050 |     0.000041 |     0.000004
   4 |        0.000126 |     0.000056 |     0.000004
   5 |        0.000659 |     0.000071 |     0.000005
   6 |        0.003547 |     0.000082 |     0.000005
   7 |        0.024672 |     0.000114 |     0.000006
   8 |        0.193076 |     0.000137 |     0.000006
   9 |        1.740500 |     0.000162 |     0.000006
  10 |       17.361407 |     0.000189 |     0.000007


> **Interpretación:**
>
> - **`determinante_recursivo`** crece factorialmente $O(n!)$, por lo que se vuelve significativamente más lento a medida que $n$ aumenta.
> - **`determinante_gauss`** crece de forma cúbica $O(n^3)$, siendo mucho más eficiente que el método recursivo para matrices grandes.
> - **`numpy.linalg.det`** es el más rápido en todos los casos, ya que utiliza implementaciones optimizadas en C (LAPACK) con algoritmos altamente eficientes.
>
> Esto confirma que la elección del algoritmo tiene un impacto drástico en el rendimiento computacional, especialmente al escalar el tamaño de los datos.

---

## Ejercicio 2 — Descenso de Gradiente

En este ejercicio trabajaremos con el método de descenso de gradiente, el cual constituye otra herramienta crucial, en esta ocasión de la rama del cálculo, para el proceso de retropropagación asociado al entrenamiento de una red neuronal.

**a) [1 punto]** Prográmese en Python el método de descenso de gradiente para funciones de $n$ variables. La función deberá tener como parámetros de entrada:

- El gradiente de la función que se desea minimizar $\nabla f$ (puede venir dada como otra función previamente implementada, `grad_f`, con entrada un vector, representando el punto donde se quiere calcular el gradiente, y salida otro vector, representando el gradiente de $f$ en dicho punto).
- Un valor inicial $x_0 \in \mathbb{R}^n$ (almacenado en un vector de $n$ componentes).
- El ratio de aprendizaje $\gamma$ (que se asume constante para cada iteración).
- Un parámetro de tolerancia `tol` (con el que finalizar el proceso cuando $\|\nabla f(x)\|_2 < $ tol).
- Un número máximo de iteraciones `maxit` (con el fin de evitar ejecuciones indefinidas en caso de divergencia o convergencia muy lenta).

La salida de la función deberá ser la aproximación del $x$ que cumple $f'(x) \approx 0$, correspondiente a la última iteración realizada en el método.

In [20]:
# a) Método de descenso de gradiente
def gradiente(x, tasa_apredizaje, tolerancia, iteraciones):
    tol = 0
    x_anterior = 3
    iter = 0 
    funcion = Polynomial([7, 0, -12, 4, 3])
    derivada = funcion.deriv()
    matriz_derivada = derivada.coef
    while iter <= iteraciones and tol <= tolerancia:
        elevado = 0
        resultado = 0
        for i in matriz_derivada:
            resultado += np.round(i*(x)**(elevado),2)
            elevado += 1
        x_anterior = x
        x = np.round(x - tasa_apredizaje * resultado,2)
        tol = np.round(x - x_anterior,2)
        iter +=1
    print(f"|Iteracion: {iter}| Punto inicial: {x_anterior}| Sustitucion: {resultado}| Aproximacion: {x}|")

def gradiente_2_variables(x0, tasa_aprendizaje, tolerancia, iteraciones):
    tol = 0
    x0_anterior = np.array([-1,1])
    iter = 0
    x, y = sp.symbols('x y')

    funcion = x**2 + y**3 + 3*x*y + 1

    df_dx = sp.diff(funcion, x)
    df_dy = sp.diff(funcion, y)

    while iter < iteraciones: #and tol < tolerancia:
        resultado_x = 0
        resultado_y = 0
        funcion_x_ejecutable = sp.lambdify((x,y),df_dx)
        funcion_y_ejecutable = sp.lambdify((x,y),df_dy)
        resultado_x = funcion_x_ejecutable(x0[0],x0[1])
        resultado_y = funcion_y_ejecutable(x0[0],x0[1])
        x0_anterior = x0
        x0 = np.array([resultado_x, resultado_y])
        
        x0[0] = x0_anterior[0] - tasa_aprendizaje * x0[0]
        x0[1] = x0_anterior[1] - tasa_aprendizaje * x0[1]
        tol = np.round(math.sqrt((x0[0]-x0_anterior[0])**2+(x0[1] - x0_anterior[1])**2),2)
        grad = resultado_x+resultado_y
        grad = math.sqrt(grad)
        iter+=1
    print(f"|Iteracion: {iter}| Aproximacion: {x0} | Gradiente: {grad}")



**b)** Sea la función $f : \mathbb{R} \to \mathbb{R}$ dada por

$$f(x) = 3x^4 + 4x^3 - 12x^2 + 7$$

> **R/**
>
> El comportamiento se da porque no hay movimiento en la gradiente. Por lo que el calculo siempre se realiza sobre el mismo punto.
>
> **Estudio analitico**
>
> Sea $$f(x) = 3x^4 + 4x^3 - 12x^2 + 7$$
>
> **a) Primera derivada**
>
> $$f(x) = 3x^4 + 4x^3 - 12x^2 + 7$$
>
> $$f'(x) = 12x^3 + 12x^2 - 24x  =  12x(x^2 + x - 2)  =  12x(x + 2)(x - 1)$$
>
> **b) Puntos críticos (Gradiente = 0)**
>
> $$12x(x + 2)(x - 1)$$
>
> $$12x = 0  =  x = 0$$
>
> $$x + 2 = 0  =  x = -2$$
>
> $$x - 1 = 0  =  x = 1$$
>
> **Puntos Criticos**
>
> $$(-2, 0, 1)$$
>
> **Segunda derivada**:
>
> $$f'(x) = 12x^3 + 12x^2 - 24x  =  36x^2 + 24x - 24 = 12(3x^2 + 2X -2)$$
>
> Sustituimos
>
> Para x = -2
> 
> $$36(-2)^2 + 24(-2) - 24 = 72$$
>
> $$72 > 0 Minimo Local$$
>
> $$36(0)^2 + 24(0) - 24 = -24$$
>
> $$-24 < 0 Maximo Local$$
>
> $$36(-2)^2 + 24(-2) - 24 = 36$$
>
> $$36 > 0 Minimo Local$$
>
> **Evaluar en f(x)**
>
> $$f(x) = 3x^4 + 4x^3 - 12x^2 + 7$$
>
> Para -2 $$3(-2)^4 + 4(-2)^3 + 12(-2)^2 + 7 = -25$$
>
> Para 0 $$3(0)^4 + 4(0)^3 + 12(0)^2 + 7 = 7$$
>
> Para 1 $$3(1)^4 + 4(1)^3 + 12(1)^2 + 7 = 2$$
>
> **Comportamiento en infinito**
>
> **Comportamiento en el infinito:**
>
> Cuando $x \to +\infty$, $3x^4 \to +\infty$ $\Rightarrow$ $f(x) \to +\infty$
>
> Cuando $x \to -\infty$, $3x^4 \to +\infty$ (el exponente es par) $\Rightarrow$ $f(x) \to +\infty$
>
> Por lo tanto, la función es acotada internamente y tiene mínimos globales.
>
> **Minimo global**
>
> $x = -2 \Rightarrow -25$
>
> $x = 1 \Rightarrow 2$
>
> $-25 < 2$ Entonces -25 es el minimo global.


**b.i) [0.5 puntos]** Aplica el método sobre $f(x)$ con $x_0 = 3$, $\gamma = 0.001$, `tol=1e-12`, `maxit=1e5`.

In [17]:
# b.i)
gradiente(3, 0.001, 0.000000000001, 99999)


|Iteracion: 100000| Punto inicial: 1.11| Sustitucion: 4.559999999999999| Aproximacion: 1.11|


**b.ii) [0.5 puntos]** Aplica de nuevo el método sobre $f(x)$ con $x_0 = 3$, $\gamma = 0.01$, `tol=1e-12`, `maxit=1e5`.

In [18]:
# b.ii)
gradiente(3, 0.01, 0.000000000001, 99999)

|Iteracion: 100000| Punto inicial: -2.0| Sustitucion: 0.0| Aproximacion: -2.0|


**b.iii) [0.5 puntos]** Contrasta e interpreta los dos resultados obtenidos en los apartados anteriores y compáralos con los mínimos locales obtenidos analíticamente. ¿Qué influencia puede llegar a tener la elección del ratio de aprendizaje $\gamma$?

> **R/**
>
>   El comportamiento observado está directamente relacionado con la tasa de aprendizaje seleccionada para el descenso de gradiente. Una tasa excesivamente alta puede provocar divergencia, impidiendo que el algoritmo converja hacia el mínimo requerido; en contraste, una tasa de aprendizaje igual a cero anula por completo cualquier actualización de los parámetros, dejando al modelo estancado sin posibilidad de mejora.
>
>   En el análisis analítico de la función $f(x) = 3x^4 + 4x^3 - 12x^2 + 7$, se determina que el mínimo global se alcanza en x=−2. Dicho valor coincide con la aproximación obtenida mediante la implementación algorítmica, lo que valida la correcta programación del método.
>
>   No obstante, al modificar la tasa de aprendizaje, se confirman los casos mencionados inicialmente: con valores extremos —como 0 o cifras muy altas— el algoritmo no logra aproximarse a la solución óptima, evidenciando la sensibilidad del método ante este hiperparámetro y la importancia de su ajuste para garantizar una convergencia estable y precisa.
> 

**b.iv) [0.5 puntos]** Aplica nuevamente el método sobre $f(x)$ con $x_0 = 3$, $\gamma = 0.1$, `tol=1e-12`, `maxit=1e5`. Interpreta el resultado.

In [11]:
# b.iv)
gradiente(3, 0.1, 0.000000000001, 99999)


|Iteracion: 1| Punto inicial: 3| Sustitucion: 360.0| Aproximacion: -33.0|
|Iteracion: 2| Punto inicial: -33.0| Sustitucion: -417384.0| Aproximacion: 41705.4|


**b.v) [0.5 puntos]** Finalmente, aplica el método sobre $f(x)$ con $x_0 = 0$, $\gamma = 0.001$, `tol=1e-12`, `maxit=1e5`. Interpreta el resultado y compáralo con el estudio analítico de $f$. ¿Se trata de un resultado deseable? ¿Por qué? ¿A qué se debe este fenómeno?

In [19]:
# b.v)
gradiente(0, 0.001, 0.000000000001, 99999)

|Iteracion: 100000| Punto inicial: 0.0| Sustitucion: 0.0| Aproximacion: 0.0|


**c)** Sea la función $g : \mathbb{R}^2 \to \mathbb{R}$ dada por

$$g(x, y) = x^2 + y^3 + 3xy + 1$$

**c.i) [0.5 puntos]** Aplíquese el método sobre $g(x, y)$ con $x_0 = (-1, 1)$, $\gamma = 0.01$, `tol=1e-12`, `maxit=1e5`.

In [21]:
# c.i)
gradiente_2_variables(np.array([-1.0,1.0]), 0.01, 0.00000000001, 99999)

|Iteracion: 99999| Aproximacion: [-2.25  1.5 ] | Gradiente: 1.0745380149674383e-07


**c.ii) [0.5 puntos]** ¿Qué ocurre si ahora partimos de $x_0 = (0, 0)$? ¿Se obtiene un resultado deseable?

In [22]:
# c.ii)
gradiente_2_variables(np.array([0.0,0.0]), 0.01, 0.00000000001, 99999)

|Iteracion: 99999| Aproximacion: [0. 0.] | Gradiente: 0.0


> **R/**
> No, porque no existe movimiento sobre la pendiente, esto hace que sea un ciclo infinito o hasta que lleguemos a las 10000 repeticiones.
> 

**c.iii) [0.5 puntos]** Realícese el estudio analítico de la función y utilícese para explicar y contrastar los resultados obtenidos en los dos apartados anteriores.

> **R/**
> Sea $$f(x, y) = x^2 + y^3 + 3xy + 1$$
>
> **a) Gradiente**
>
> $$\frac{\partial f}{\partial x} = 2x + 3y \quad \text{("y" se comporta como una constante)}$$
>
> $$\frac{\partial f}{\partial y} = 3y^2 + 3x \quad \text{("x" se comporta como una constante)}$$
>
> $$\nabla f(x,y) = (2x + 3y,\; 3y^2 + 3x)$$
>
> **b) Puntos críticos (Gradiente = 0)**
>
> $$2x + 3y = 0$$
>
> $$2x = -3y$$
>
> $$x = \frac{-3y}{2}$$
>
> $$3y^2 + 3x = 0$$
>
> Sustituyendo:
>
> $$3y^2 + 3\left(\frac{-3y}{2}\right) = 0$$
>
> $$3y^2 - \frac{9y}{2} = 0$$
>
> $$3y\left(y - \frac{3}{2}\right) = 0$$
>
> **Entonces:**
>
> $$3y = 0 \Rightarrow y = 0$$
>
> $$y - \frac{3}{2} = 0 \Rightarrow y = \frac{3}{2}$$
>
> **Para y = 0:**
>
> $$x = \frac{-3(0)}{2} = 0$$
>
> **Para y = 3/2:**
>
> $$x = \frac{-3}{2} \cdot \frac{3}{2} = \frac{-9}{4}$$
>
> **Puntos críticos:**
>
> $$(0, 0) \quad \text{y} \quad \left(\frac{-9}{4}, \frac{3}{2}\right)$$
> 
> **Matriz Heussiana:**
>
> $$\nabla f''(x,x) = 2x + 3y = 2$$
>
> $$\nabla f''(x,y) = 2x + 3y = 3$$
>
> $$\nabla f''(y,x) = 3y^2 + 3x = 3$$
>
> $$\nabla f''(y,y) = 3y^2 + 3x = 6y$$
>
> $$H_f(x,y) = \begin{pmatrix}
> 2 & 3 \\
> 3 & 6y
> \end{pmatrix}$$
>
> **Evaluamos en el punto:**
> $$(0, 0) \quad \text{y} \quad \left(\frac{-9}{4}, \frac{3}{2}\right)$$
>
> $$H_f(x,y) = \begin{pmatrix}
> 2 & 3 \\
> 3 & 6y
> \end{pmatrix}$$
>
> $$H_f(x,y) = \begin{pmatrix}
> 2 & 3 \\
> 3 & 6y
> \end{pmatrix}$$
>
> **Evaluamos en el punto** $(0, 0)$**:**
>
> $$H_f(0,0) = \begin{pmatrix}
> 2 & 3 \\
> 3 & 0
> \end{pmatrix}$$
>
> $$\det = 2 \cdot 0 - 3 \cdot 3 = 0 - 9 = -9$$
>
> **Punto de silla porque** $-9 < 0$
>
> **Evaluamos en el punto** $\left(\frac{-9}{4}, \frac{3}{2}\right)$**:**
>
> $$H_f\left(\frac{-9}{4}, \frac{3}{2}\right) = \begin{pmatrix}
> 2 & 3 \\
> 3 & 9
> \end{pmatrix}$$
>
> $$\det = 2 \cdot 9 - 3 \cdot 3 = 18 - 9 = 9$$
>
> **Mínimo local porque** $9 > 0$
> **Valor de la función en el mínimo:**
>
> $$f\left(\frac{-9}{4}, \frac{3}{2}\right) = \left(\frac{-9}{4}\right)^2 + \left(\frac{3}{2}\right)^3 + 3\left(\frac{-9}{4}\right)\left(\frac{3}{2}\right) + 1$$
>
> $$= \frac{81}{16} + \frac{27}{8} - \frac{81}{8} + 1$$
>
> $$= \frac{81}{16} + \frac{54}{16} - \frac{162}{16} + \frac{16}{16}$$
>
> $$= \frac{81 + 54 - 162 + 16}{16} = \frac{-11}{16}$$
>
> $$\approx -0.6875$$
>
> **Por lo tanto, el valor mínimo de la función es** $\frac{-11}{16} \approx -0.6875$


> R/
>
>   El comportamiento observado depende fundamentalmente de la tasa de aprendizaje empleada en el descenso de gradiente. Una tasa excesivamente alta puede generar inestabilidad y divergencia, impidiendo que el algoritmo alcance el punto óptimo; por el contrario, una tasa igual a cero anula cualquier actualización de los parámetros, dejando al modelo estancado sin evolución alguna.
>
>   En el caso de la función g(x,y)=x2+y3+3xy+1g(x,y)=x2+y3+3xy+1, el análisis analítico revela que el punto crítico se localiza en el intervalo (−94,32)(−49​,23​), coincidiendo con la solución obtenida mediante la implementación algorítmica, lo que confirma la validez del método programado.
>
>   No obstante, al variar la tasa de aprendizaje, se ratifican los escenarios ya señalados: con valores extremos —como 0 o magnitudes muy grandes— el algoritmo no logra converger adecuadamente. Además, se evidencia que el éxito de la convergencia también depende del punto de partida, ya que distintas inicializaciones pueden llevar a resultados divergentes o a mínimos locales, subrayando la sensibilidad del método tanto al hiperparámetro como a la condición inicial.

## Acceso a repositorio utilizado de github https://github.com/ElwingCR/Matematica